In [1]:
import pandas as pd
from sqlalchemy import create_engine
from prophet import Prophet
import plotly.graph_objects as go
import os

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
DB_PATH = os.path.join(BASE_DIR, 'data', 'stock_data.db')
engine = create_engine(f'sqlite:///{DB_PATH}')

df = pd.read_sql("SELECT * FROM stock_prices WHERE ticker = 'AAPL'", con=engine)
df['date'] = pd.to_datetime(df['date'])

prophet_df = df[['date', 'close']].rename(columns={'date': 'ds', 'close': 'y'})

print(prophet_df.shape)
print(prophet_df.tail())

(471, 2)
            ds           y
466 2026-05-22  308.820007
467 2026-05-26  308.329987
468 2026-05-27  310.850006
469 2026-05-28  312.510010
470 2026-05-29  312.059998


Train

In [4]:
model = Prophet(
    daily_seasonality = False,
    weekly_seasonality=True,
    yearly_seasonality=True,
    changepoint_prior_scale=0.05
)
model.fit(prophet_df)

future = model.make_future_dataframe(periods=90)
forecast = model.predict(future)

print(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(10))


12:57:50 - cmdstanpy - INFO - Chain [1] start processing
12:57:51 - cmdstanpy - INFO - Chain [1] done processing


            ds        yhat  yhat_lower  yhat_upper
551 2026-08-18  374.626007  360.857347  389.754935
552 2026-08-19  376.124065  361.918373  390.426942
553 2026-08-20  376.785998  362.409455  391.859775
554 2026-08-21  377.892569  363.600635  393.033448
555 2026-08-22  379.590919  365.203151  394.941501
556 2026-08-23  380.456302  366.593715  395.926076
557 2026-08-24  380.597438  365.850650  396.990678
558 2026-08-25  381.395644  365.480752  397.494599
559 2026-08-26  382.255392  367.249481  398.118121
560 2026-08-27  382.313909  366.736715  399.116144


In [7]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=prophet_df['ds'], y=prophet_df['y'],
    name='Actual', line=dict(color='gray', width=1)
))

fig.add_trace(go.Scatter(
    x=forecast['ds'], y=forecast['yhat'],
    name='Predicted', line=dict(color='blue', width=2)
))

fig.add_trace(go.Scatter(
    x=list(forecast['ds']) + list(forecast['ds'][::-1]),
    y=list(forecast['yhat_upper']) + list(forecast['yhat_lower'][::-1]),
    fill='toself',
    fillcolor='rgba(0,100,255,0.1)',
    line=dict(color='rgba(255,255,255,0)'),
    name='Confidence Interval'
))

split_date = prophet_df['ds'].max().to_pydatetime()
fig.add_vline(
    x=split_date.timestamp() * 1000,
    line_dash='dash',
    line_color='red',
    annotation_text='Forecast Start'
)

fig.update_layout(
    title='AAPL Stock Price Prediction — Next 90 Days (Prophet)',
    xaxis_title='Date',
    yaxis_title='Price (USD)'
)

fig.show()